In [ ]:
# --- Xray Cloud: Test Plan Rollup + Daily Snapshot + Emoji/PNG Progress ---
import requests, getpass, json, collections, csv, os
from datetime import datetime

PLAN_KEY = "PROJ-1003"  # <-- set your plan key

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"

# TEMP: bypass SSL verification while testing (set True for prod)
VERIFY_SSL = False
if not VERIFY_SSL:
    from urllib3.exceptions import InsecureRequestWarning
    import urllib3; urllib3.disable_warnings(InsecureRequestWarning)

# --- Plotting is optional (emoji output works without it) ---
try:
    import matplotlib
    matplotlib.use("Agg")  # safe for headless/CI and fine on desktop
    import matplotlib.pyplot as plt
    import numpy as np
    HAS_MPL = True
except Exception as e:
    HAS_MPL = False
    print(f"[i] Plotting disabled (matplotlib not available: {e})")

# -------------------- Xray helpers --------------------
def auth(client_id: str, client_secret: str) -> str:
    r = requests.post(XRAY_AUTH_URL, json={"client_id": client_id, "client_secret": client_secret},
                      timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"Auth failed: {r.status_code} {r.text[:300]}")
    return r.text.strip('"')

def gql(query: str, variables=None, headers=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                      headers=headers or {}, timeout=60, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"GraphQL HTTP {r.status_code}: {r.text[:300]}")
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# -------------------- GraphQL queries --------------------
Q_PLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    total start limit
    results { issueId jira(fields:["key","summary"]) }
  }
}
"""

Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""

Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results { status { name } }
  }
}
"""

# -------------------- CSV + Reporting helpers --------------------
def save_daily_snapshot(plan_key: str, totals: dict) -> str:
    """Append snapshot row and include derived columns."""
    today = datetime.now().strftime("%Y-%m-%d")
    fname = f"xray_progress_{plan_key}.csv"
    row = {
        "date": today,
        "total": totals["Total"],
        "passed": totals["Passed"],
        "failed": totals["Failed"],
        "descoped": totals["Descoped"],
        "todo": totals["TO DO"],
    }
    row["completed"] = row["passed"] + row["failed"] + row["descoped"]
    row["pass_rate"] = round(100.0 * row["passed"] / row["total"], 1) if row["total"] else 0.0

    file_exists = os.path.exists(fname)
    fieldnames = ["date","total","passed","failed","descoped","todo","completed","pass_rate"]
    with open(fname, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            w.writeheader()
        w.writerow(row)
    return fname

def print_emoji_progress(csv_file: str, plan_name: str, bar_width: int = 40):
    """Print a clean text/emoji progress block (paste to Teams/Outlook/Jira)."""
    rows = []
    with open(csv_file, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            # coerce types
            r["total"] = int(float(r.get("total", 0) or 0))
            r["passed"] = int(float(r.get("passed", 0) or 0))
            r["failed"] = int(float(r.get("failed", 0) or 0))
            r["descoped"] = int(float(r.get("descoped", 0) or 0))
            r["todo"] = int(float(r.get("todo", 0) or 0))
            r["completed"] = int(float(r.get("completed", 0) or 0))
            r["pass_rate"] = float(r.get("pass_rate", 0))
            r["date"] = datetime.strptime(r["date"], "%Y-%m-%d").date()
            rows.append(r)
    if not rows:
        print("[!] No data to render.")
        return
    rows.sort(key=lambda r: r["date"])

    E_PASS, E_FAIL, E_TODO, E_DSCP = "🟩", "🟥", "🟨", "⬜"

    def bar(p, f, t, d, total):
        total = max(total, 1)
        p_n = round(bar_width * p / total)
        f_n = round(bar_width * f / total)
        t_n = round(bar_width * t / total)
        d_n = round(bar_width * d / total)
        drift = bar_width - (p_n + f_n + t_n + d_n)
        segs = [("p", p_n), ("t", t_n), ("f", f_n), ("d", d_n)]
        segs.sort(key=lambda x: x[1], reverse=True)
        segs[0] = (segs[0][0], segs[0][1] + drift)
        counts = {k:v for k,v in segs}
        return E_PASS*counts["p"] + E_FAIL*counts["f"] + E_TODO*counts["t"] + E_DSCP*counts["d"]

    latest = rows[-1]
    print(f"\n**Daily Test Execution Progress — {plan_name}**")
    print(f"_Pass rate: {latest['pass_rate']:.1f}%  |  Total runs: {latest['total']}_")
    print("Legend: 🟩 Pass  🟥 Fail  🟨 To-Do  ⬜ Descoped\n")
    for r in rows:
        b = bar(r["passed"], r["failed"], r["todo"], r["descoped"], r["total"])
        print(f"{r['date'].strftime('%b %d')}: {b}  "
              f"(P:{r['passed']} F:{r['failed']} T:{r['todo']} D:{r['descoped']})")

def plot_progress(csv_file: str, plan_key: str, plan_name: str):
    """Create PNG: Complete = Pass(green)+Fail(red); To-Do yellow; Descoped grey; thin Total line."""
    if not HAS_MPL:
        return None
    dates, total, passed, failed, todo, descoped = [], [], [], [], [], []
    with open(csv_file, newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            dates.append(datetime.strptime(row["date"], "%Y-%m-%d").date())
            total.append(int(row["total"]))
            passed.append(int(row["passed"]))
            failed.append(int(row["failed"]))
            todo.append(int(row["todo"]))
            descoped.append(int(row["descoped"]))
    if not dates:
        print("[!] No data to plot yet.")
        return None

    # sort by date
    idx = sorted(range(len(dates)), key=lambda i: dates[i])
    dates    = [dates[i]    for i in idx]
    total    = [total[i]    for i in idx]
    passed   = [passed[i]   for i in idx]
    failed   = [failed[i]   for i in idx]
    todo     = [todo[i]     for i in idx]
    descoped = [descoped[i] for i in idx]

    x = np.arange(len(dates))
    bar_width = 0.6
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.bar(x, passed,  width=bar_width, label="Passed (Complete)",  color="green")
    ax.bar(x, failed,  bottom=passed,   width=bar_width, label="Failed (Complete)",  color="red")
    ax.bar(x, todo,    bottom=np.array(passed)+np.array(failed), width=bar_width, label="To-Do",    color="yellow")
    ax.bar(x, descoped,bottom=np.array(passed)+np.array(failed)+np.array(todo),
           width=bar_width, label="Descoped", color="grey")

    ax.plot(x, total, marker="x", linestyle="-", linewidth=1, label="Total (Target)")

    ax.set_xticks(x)
    ax.set_xticklabels([d.strftime("%b %d") for d in dates])
    ax.set_xlabel("Date")
    ax.set_ylabel("Test Runs (daily snapshot)")
    latest_pr = (passed[-1] / total[-1] * 100) if total[-1] else 0.0
    ax.set_title(f"Daily Test Execution Progress — {plan_name}  |  Pass rate: {latest_pr:.1f}%")
    ax.legend(loc="upper left", ncol=3, fontsize=8)
    fig.tight_layout()

    out_png = f"xray_progress_{plan_key}.png"
    fig.savefig(out_png, dpi=150)
    plt.close(fig)
    return out_png

# -------------------- Main flow --------------------
if __name__ == "__main__":
    # 1) Auth
    CID  = getpass.getpass("XRAY_CLIENT_ID (Xray → API Keys): ")
    CSEC = getpass.getpass("XRAY_CLIENT_SECRET (Xray → API Keys): ")
    TOKEN   = auth(CID, CSEC)
    HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

    # 2) Resolve plan key -> issueId
    plans = gql(Q_PLANS, {"jql": f'key = "{PLAN_KEY}"', "limit": 1, "start": 0}, headers=HEADERS)
    results = plans["getTestPlans"]["results"]
    if not results:
        raise SystemExit(f"No Test Plan found for key {PLAN_KEY}")
    PLAN_ID = results[0]["issueId"]
    PLAN_NAME = f'{results[0]["jira"]["key"]} - {results[0]["jira"]["summary"]}'

    # 3) Collect Test Execution ids (paged)
    exec_ids, start, page = [], 0, 100
    while True:
        d = gql(Q_PLAN_EXECS, {"planId": PLAN_ID, "limit": page, "start": start}, headers=HEADERS)
        te = d["getTestPlan"]["testExecutions"]
        exec_ids += [x["issueId"] for x in te["results"]]
        start += te["limit"]
        if start >= te["total"]:
            break

    # 4) Page Test Runs in batches, count statuses
    counts = collections.Counter()
    total_runs = 0
    BATCH = 50
    for i in range(0, len(exec_ids) or 1, max(1, BATCH)):  # handle empty plans
        batch = exec_ids[i:i+BATCH]
        start = 0
        limit = 100
        while True:
            data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start}, headers=HEADERS)
            runs = data["getTestRuns"]
            for tr in runs["results"]:
                sname = (tr.get("status") or {}).get("name") or "UNKNOWN"
                counts[sname] += 1
                total_runs += 1
            start += runs["limit"]
            if start >= runs["total"]:
                break

    # 5) Normalize into buckets
    PASSED    = sum(counts.get(k, 0) for k in ("PASSED", "Pass", "OK"))
    FAILED    = sum(counts.get(k, 0) for k in ("FAILED", "Fail"))
    DESCOPED  = sum(counts.get(k, 0) for k in ("DESCOPED", "Not Executed (Descoped)", "SKIPPED (Descoped)"))
    TODO      = sum(counts.get(k, 0) for k in ("TODO", "TO DO", "NOT_RUN", "Untested", "To Do"))

    summary = {
        "Test Plan": PLAN_NAME,
        "Totals (Test Runs)": {
            "Total": total_runs,
            "Passed": PASSED,
            "Failed": FAILED,
            "Descoped": DESCOPED,
            "TO DO": TODO
        },
        "Other Statuses": {k: v for k, v in counts.items()
                           if k not in {"PASSED","Pass","OK","FAILED","Fail",
                                        "DESCOPED","Not Executed (Descoped)","SKIPPED (Descoped)",
                                        "TODO","TO DO","To Do","NOT_RUN","Untested"}}
    }

    # 6) Print JSON + Markdown rollup (as before)
    print(json.dumps(summary, indent=2))
    print("\n".join([
        f"\n### Test Plan Rollup — {PLAN_NAME}\n",
        "| Metric | Count |",
        "|---|---|",
        f"| Total | {summary['Totals (Test Runs)']['Total']} |",
        f"| Passed | {summary['Totals (Test Runs)']['Passed']} |",
        f"| Failed | {summary['Totals (Test Runs)']['Failed']} |",
        f"| Descoped | {summary['Totals (Test Runs)']['Descoped']} |",
        f"| To Do | {summary['Totals (Test Runs)']['TO DO']} |",
    ]))

    # 7) Append daily snapshot
    csv_path = save_daily_snapshot(PLAN_KEY, summary["Totals (Test Runs)"])
    print(f"\n[+] Snapshot appended to {csv_path}")

    # 8) Print emoji block (always works)
    print_emoji_progress(csv_path, PLAN_NAME)

    # 9) Save PNG chart (only if matplotlib is available)
    png_path = plot_progress(csv_path, PLAN_KEY, PLAN_NAME)
    if png_path:
        print(f"\n[+] Saved chart: {png_path}")
    else:
        print("\n[i] Skipping PNG chart (matplotlib not installed or unavailable).")
